# Chunks de los textos recuperados

In [1]:
from pathlib import Path
import json

In [2]:
INPUT_DIR = Path("../shared/output/financiera/textos")
OUTPUT_DIR = Path("../shared/output/financiera/chunks")

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

def crear_chunks(texto, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    texto = texto.strip()

    chunks = []

    inicio = 0

    while inicio < len(texto):
        fin = inicio + chunk_size

        chunk = texto[inicio:fin].strip()

        if chunk:
            chunks.append(chunk)

        inicio = fin - overlap

    return chunks

def procesar_contratos():
    resultados = []

    contratos_dir = INPUT_DIR / "contratos"

    for archivo in sorted(contratos_dir.glob("*.txt")):
        contrato = archivo.stem

        texto = archivo.read_text(encoding="utf-8")

        chunks = crear_chunks(texto)

        for i, chunk in enumerate(chunks, start=1):
            resultados.append({
                "id": f"contrato-{contrato}-{i:04d}",
                "contrato": contrato,
                "tipo": "contrato",
                "archivo": archivo.name,
                "chunk": i,
                "texto": chunk
            })

    return resultados

def procesar_pagos():
    resultados = []

    pagos_dir = INPUT_DIR / "pagos"

    for contrato_dir in sorted(pagos_dir.iterdir()):
        if not contrato_dir.is_dir():
            continue

        contrato = contrato_dir.name

        for archivo in sorted(contrato_dir.glob("*.txt")):
            texto = archivo.read_text(encoding="utf-8")

            chunks = crear_chunks(texto)

            for i, chunk in enumerate(chunks, start=1):
                resultados.append({
                    "id": f"pago-{contrato}-{archivo.stem}-{i:04d}",
                    "contrato": contrato,
                    "tipo": "pago",
                    "archivo": archivo.name,
                    "chunk": i,
                    "texto": chunk
                })

    return resultados

def guardar_json(path, datos):
    path.parent.mkdir(parents=True, exist_ok=True)

    path.write_text(
        json.dumps(
            datos,
            indent=4,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

In [3]:
contratos = procesar_contratos()
pagos = procesar_pagos()

todos = contratos + pagos

guardar_json(
    OUTPUT_DIR / "contratos.json",
    contratos
)

guardar_json(
    OUTPUT_DIR / "pagos.json",
    pagos
)

guardar_json(
    OUTPUT_DIR / "todos.json",
    todos
)

print(f"Chunks contratos: {len(contratos)}")
print(f"Chunks pagos:     {len(pagos)}")
print(f"Chunks totales:   {len(todos)}")

Chunks contratos: 96
Chunks pagos:     98
Chunks totales:   194
